## 1. Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
from sklearn.linear_model import LassoCV, LinearRegression, RidgeCV
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm

CONFIG = {
    "validation_start": "2012-02-01",  # the first day with 250 rows of features before it
    "validation_end": "2017-12-31",
    "test_start": "2018-01-01",
    "test_end": "2025-12-31",
    "train_windows": [125, 250],  # training window L, in trading days
    "refit_intervals": [5, 10, 21],  # trading days between refits
    "embargo_rows": 2,  # row t-1's label needs Open[t+1], unknown at the close of t
    "cv_folds": 5,
    "lasso_n_penalties": 20,
    "lasso_min_penalty_ratio": 0.05,  # penalty grid: from the largest (keeps nothing) down to 5% of it
    "ridge_penalties": [float(penalty) for penalty in np.logspace(-1, 5, 13)],
    "ar_lags": 3,
    "feature_clip": 5.0,  # standardised features are clipped to +-5
    "label_winsor_quantiles": [0.01, 0.99],  # for train labels winsorization
    "n_jobs": -2,
}

DATA_DIR = Path("./data/prepared")
OUTPUT_DIR = Path("./results")

OUTPUT_DIR.mkdir(exist_ok=True)

MODELS = ["lasso", "ridge", "ar", "mean"]
MODEL_NAMES = {
    "lasso": "LASSO",
    "ridge": "Ridge",
    "ar": "AR(3)",
    "mean": "Historical mean",
}

## 2. Features and labels

In [ ]:
MODEL_DATA_FILE = DATA_DIR / "model_data.csv"
FEATURE_COLUMNS = [
    "return_1d",
    "return_5d",
    "return_20d",
    "ma_gap_20d",
    "volatility_20d",
    "intraday_range",
    "volume_ratio_20d",
]


model_data = pd.read_csv(
    MODEL_DATA_FILE, parse_dates=["Date", "entry_date", "exit_date"]
)
TICKERS = sorted(model_data["Ticker"].unique())
N_STOCKS = len(TICKERS)


def to_wide_frame(column):
    return model_data.pivot(index="Date", columns="Ticker", values=column)[
        TICKERS
    ].sort_index()


SIGNAL_DATES = to_wide_frame("target_return").index
OWN_FEATURES = [feature for feature in FEATURE_COLUMNS if feature != "return_1d"]
FEATURES = {feature: to_wide_frame(feature) for feature in FEATURE_COLUMNS}
LABELS = to_wide_frame("target_return")
AR_INPUTS = {
    f"label_lag{lag}": LABELS.shift(lag) for lag in range(2, 2 + CONFIG["ar_lags"])
}


def to_float_array(frame):
    values = frame.to_numpy(dtype=float, copy=True)
    values[~np.isfinite(values)] = np.nan
    return values


CROSS_RETURNS = to_float_array(FEATURES["return_1d"])
OWN_INDICATORS = np.stack(
    [to_float_array(FEATURES[feature]) for feature in OWN_FEATURES], axis=2
)
AR_RETURNS = np.stack(
    [to_float_array(AR_INPUTS[lag_name]) for lag_name in sorted(AR_INPUTS)], axis=2
)
LABEL_ARRAY = to_float_array(LABELS)


FIRST_REFIT_ROW = int(
    SIGNAL_DATES.searchsorted(pd.Timestamp(CONFIG["validation_start"]))
)
FORECAST_END_ROW = int(
    SIGNAL_DATES.searchsorted(pd.Timestamp(CONFIG["test_end"]), side="right")
)
REFIT_ROWS = {
    refit_interval: np.arange(FIRST_REFIT_ROW, FORECAST_END_ROW, refit_interval)
    for refit_interval in CONFIG["refit_intervals"]
}

assert (
    np.isfinite(CROSS_RETURNS).all()
    and np.isfinite(OWN_INDICATORS).all()
    and np.isfinite(LABEL_ARRAY).all()
)

## 3. Rolling estimation

In [ ]:
def run_stock(
    model,
    window,
    config,
    cross_returns,
    own_indicators,
    ar_returns,
    labels,
    refit_rows,
    end_row,
):
    """Rolling fits for one stock; returns its daily forecasts (NaN where no model was fitted)."""

    # Set features for each model.
    features = None
    if model == "lasso" or model == "ridge":
        # LASSO and ridge see every stock's return plus this stock's indicators
        features = np.hstack([cross_returns, own_indicators])
    elif model == "ar":
        # AR(3) sees only this stock's last three known labels
        features = ar_returns
    elif model == "mean":
        # Historical mean needs none
        pass

    # Forward folds for choosing the penalty with a gap as long as the embargo.
    cv_folds = TimeSeriesSplit(n_splits=config["cv_folds"], gap=config["embargo_rows"])
    forecasts = np.full(len(labels), np.nan)

    for refit_index, refit_row in enumerate(refit_rows):
        # --- 1. Training rows: the L rows that end just before the refit (the embargo) ---
        last_train_row = refit_row - config["embargo_rows"]
        first_train_row = last_train_row - window + 1
        if first_train_row < 0:
            # not enough history yet
            continue
        train_rows = np.arange(first_train_row, last_train_row + 1)

        # --- 2. Forecast rows: from this refit up to the next one; the coefficients stay frozen ---
        next_refit_row = (
            refit_rows[refit_index + 1]
            if refit_index + 1 < len(refit_rows)
            else end_row
        )
        forecast_rows = np.arange(refit_row, next_refit_row)

        # --- 3. Training labels, winsorised at the window's 1st and 99th percentiles ---
        train_labels = np.clip(
            labels[train_rows],
            *np.quantile(labels[train_rows], config["label_winsor_quantiles"])
        )

        # Historical mean: the forecast is simply the average training label.
        if model == "mean":
            forecasts[forecast_rows] = train_labels.mean()
            continue

        # --- 4. Scale with training-window statistics, then fit and forecast ---
        scaler = StandardScaler().fit(features[train_rows])
        train_scaled = np.nan_to_num(
            np.clip(
                scaler.transform(features[train_rows]), 
                -config["feature_clip"], 
                config["feature_clip"]
            ), nan=0.0
        )
        forecast_scaled = np.nan_to_num(
            np.clip(
                scaler.transform(features[forecast_rows]), 
                -config["feature_clip"], 
                config["feature_clip"]
            ), nan=0.0
        )

        if model == "lasso":
            # Penalty chosen by forward-fold CV over 20 values, from the largest (keeps nothing)
            # down to 5% of it.
            fitted = LassoCV(
                alphas=config["lasso_n_penalties"],
                eps=config["lasso_min_penalty_ratio"],
                cv=cv_folds,
                precompute=True,
                max_iter=5000,
            ).fit(train_scaled, train_labels)
        elif model == "ridge":
            # Same folds, scored by mean squared error, as LassoCV is.
            fitted = RidgeCV(
                alphas=config["ridge_penalties"],
                cv=cv_folds,
                scoring="neg_mean_squared_error",
            ).fit(train_scaled, train_labels)
        else:
            # AR(3): ordinary least squares on the stock's own last three known labels.
            fitted = LinearRegression().fit(train_scaled, train_labels)
        forecasts[forecast_rows] = fitted.predict(forecast_scaled)

    return forecasts


# One forecast frame (signal dates x stocks) per model, window and refit interval.
# The stocks are fitted in parallel.
fit_settings = [
    (model, window, refit_interval)
    for model in MODELS
    for window in CONFIG["train_windows"]
    for refit_interval in CONFIG["refit_intervals"]
]
FORECASTS = {}
for model, window, refit_interval in tqdm(fit_settings, desc="fits"):
    per_stock_forecasts = Parallel(n_jobs=CONFIG["n_jobs"])(
        delayed(run_stock)(
            model,
            window,
            CONFIG,
            CROSS_RETURNS,
            OWN_INDICATORS[:, stock_index, :],
            AR_RETURNS[:, stock_index, :],
            LABEL_ARRAY[:, stock_index],
            REFIT_ROWS[refit_interval],
            FORECAST_END_ROW,
        )
        for stock_index in range(N_STOCKS)
    )
    FORECASTS[(model, window, refit_interval)] = pd.DataFrame(
        np.column_stack(per_stock_forecasts), index=SIGNAL_DATES, columns=TICKERS
    )

## 4. Forecast accuracy

In [ ]:
def r2(forecasts, dates, benchmark=None):
    realised = LABELS.loc[dates].to_numpy()
    predicted = forecasts.loc[dates].to_numpy()

    usable = np.isfinite(realised) & np.isfinite(predicted)
    if benchmark is not None:
        benchmark_values = benchmark.loc[dates].to_numpy()
        usable &= np.isfinite(benchmark_values)

    if benchmark is None:
        denominator = (realised[usable] ** 2).sum()
    else:
        denominator = ((realised - benchmark_values)[usable] ** 2).sum()

    return 1 - ((realised - predicted)[usable] ** 2).sum() / denominator

In [ ]:
VALIDATION_DATES = SIGNAL_DATES[
    (SIGNAL_DATES >= pd.Timestamp(CONFIG["validation_start"]))
    & (SIGNAL_DATES <= pd.Timestamp(CONFIG["validation_end"]))
]
print(
    f"Validation: {VALIDATION_DATES[0].date()} to {VALIDATION_DATES[-1].date()} ({len(VALIDATION_DATES)} days)"
)

table_rows = []
for (model, window, refit_interval), forecasts in FORECASTS.items():
    mean_forecasts = FORECASTS[("mean", window, refit_interval)]
    table_rows.append(
        {
            "model": MODEL_NAMES[model],
            "L": window,
            "R (days)": refit_interval,
            "R2 vs zero (%)": 100 * r2(forecasts, VALIDATION_DATES),
            "R2 vs hist. mean (%)": (
                100 * r2(forecasts, VALIDATION_DATES, mean_forecasts)
            ),
        }
    )
pd.DataFrame(table_rows).set_index(["model", "L", "R (days)"]).round(4)

In [ ]:
# Best window length and refit interval for each model
CHOSEN = {
    "lasso": (250, 5),
    "ridge": (250, 21),
    "ar": (250, 21),
    "mean": (250, 5),
}

In [ ]:
TEST_DATES = SIGNAL_DATES[
    (SIGNAL_DATES >= pd.Timestamp(CONFIG["test_start"]))
    & (SIGNAL_DATES <= pd.Timestamp(CONFIG["test_end"]))
]
print(
    f"Test: {TEST_DATES[0].date()} to {TEST_DATES[-1].date()} ({len(TEST_DATES)} days)"
)

benchmark_forecasts = FORECASTS[("mean", *CHOSEN["mean"])]
table_rows = []
for model, (window, refit_interval) in CHOSEN.items():
    forecasts = FORECASTS[(model, window, refit_interval)]
    table_rows.append(
        {
            "model": MODEL_NAMES[model],
            "L": window,
            "R (days)": refit_interval,
            "R2 vs zero (%)": 100 * r2(forecasts, TEST_DATES),
            "R2 vs hist. mean (%)": (
                100 * r2(forecasts, TEST_DATES, benchmark_forecasts)
            ),
        }
    )
pd.DataFrame(table_rows).set_index(["model", "L", "R (days)"]).round(4)

In [ ]:
# Test-period R2 vs zero by calendar year, 
# each model at its own chosen setting
test_years = sorted(set(TEST_DATES.year))
r2_by_year = pd.DataFrame(
    {
        MODEL_NAMES[model]: [
            100
            * r2(
                FORECASTS[(model, *CHOSEN[model])],
                TEST_DATES[TEST_DATES.year == year],
            )
            for year in test_years
        ]
        for model in MODELS
    },
    index=pd.Index(test_years, name="year"),
)

BAR_COLORS = {
    "lasso": "#2a78d6",
    "ridge": "#eb6834",
    "ar": "#1baf7a",
    "mean": "#898781",
}
bar_width = 0.13
bar_gap = 0.015
positions = np.arange(len(test_years))

fig, ax = plt.subplots(figsize=(8, 4.2))
for k, model in enumerate(MODELS):
    offset = (k - (len(MODELS) - 1) / 2) * (bar_width + bar_gap)
    window, refit_interval = CHOSEN[model]
    ax.bar(
        positions + offset,
        r2_by_year[MODEL_NAMES[model]],
        width=bar_width,
        color=BAR_COLORS[model],
        label=f"{MODEL_NAMES[model]} (L = {window}, R = {refit_interval})",
        zorder=2,
    )
ax.grid(axis="y", color="#e1e0d9", linewidth=0.8)
ax.set_axisbelow(True)
ax.set_xticks(positions, test_years)
ax.tick_params(length=0)
ax.set_xlabel("Year")
ax.set_ylabel("R² vs zero forecast (%)")
ax.set_title("Test period: out-of-sample R² by year", pad=24)
ax.legend(
    loc="lower center",
    bbox_to_anchor=(0.5, 1.0),
    ncols=4,
    frameon=False,
    fontsize=7,
)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "test_r2_by_year.png", dpi=200)
# plt.show()

## 5. Export LASSO Forecasts for Trading Evaluation

In [ ]:
key = ("lasso", 250, 5)
f = FORECASTS[key].rename_axis("Date")

forecast_long = f.reset_index().melt(
    id_vars="Date", var_name="Ticker", value_name="predicted_return"
)

output_path = Path("data/prepared/forecasts.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

forecast_long.dropna(subset=["predicted_return"]).to_csv(output_path, index=False)

print(f"Forecasts exported to {output_path}")

## 6. Trading Strategy and Out-of-Sample Evaluation

In [ ]:
import subprocess
import sys
import pandas as pd
from IPython.display import display, Image

# Run trading evaluation
subprocess.run(
    [sys.executable, "evaluation.py"], check=True, capture_output=True, text=True
)

# Display selected parameters and validation grid
grid = pd.read_csv("results/validation_grid.csv")
print("Trading Parameter Selection (Validation)")
display(grid.sort_values("sharpe_rf0", ascending=False).head())

# Display performance metrics
performance = pd.read_csv("results/performance.csv")
print("\nTrading Strategy Performance")
display(performance.round(4))

# Display cumulative wealth plots
print("\nValidation Cumulative Wealth")
display(Image(filename="results/validation_cumulative.png"))

print("\nTest Cumulative Wealth")
display(Image(filename="results/test_cumulative.png"))